# Evaluacion de retrieval con RAGAS (precision y recall de contexto)

Toma los archivos de retrieval generados por los dos notebooks anteriores (`resultados_retrieval.json` con las 9 configuraciones dense-only, y `resultados_retrieval_bge_m3_hibrido.json` con la configuracion hibrida) y calcula, para cada una, metricas de **precision** y **recall** de los contextos recuperados frente a las 296 preguntas del dataset de validacion.

Solo se evalua retrieval (no generacion): no hay `answer` generado todavia, asi que las metricas de RAGAS que necesitan una respuesta (faithfulness, answer_relevancy) no aplican aun. Eso queda para una siguiente etapa.

**Resiliencia frente a caidas de OpenRouter** (en 3 capas, de la mas a la menos automatica):
1. Fallback de modelos de OpenRouter (`models=[...]` en la peticion): si el modelo juez principal esta caido, OpenRouter prueba otro automaticamente.
2. Reintentos propios de RAGAS (`RunConfig`, con backoff exponencial via `tenacity`) para llamadas individuales que fallan.
3. Reintentos + checkpoint por configuracion en este notebook: si una configuracion completa falla tras varios intentos, se guarda lo que ya se evaluo (en disco) y se sigue con la siguiente, sin perder el trabajo ya hecho.


In [ ]:
%pip install -q ragas langchain-openai rapidfuzz pandas

In [37]:
import json
from pathlib import Path

# Archivos de entrada (salida de los notebooks de retrieval)
RUTA_RESULTADOS_PRINCIPAL = "./validacion/resultados_retrieval.json"
RUTA_RESULTADOS_HIBRIDO = "./validacion/resultados_retrieval_bge_m3_hibrido.json"

# Archivos de salida
RUTA_SALIDA_DETALLE = "./validacion/evaluacion_ragas_detalle.csv"   # una fila por pregunta x configuracion
RUTA_SALIDA_RESUMEN = "./validacion/evaluacion_ragas_resumen.csv"   # promedio por configuracion

# Checkpoints: un CSV por configuracion ya evaluada, para no perder trabajo si algo falla
RUTA_CHECKPOINTS = Path("./validacion/checkpoints_evaluacion")
RUTA_CHECKPOINTS.mkdir(parents=True, exist_ok=True)

# None = evalua las 296 preguntas de cada configuracion.
# Pon un numero pequeño (ej. 20) para una corrida de prueba rapida y barata
# antes de lanzar la evaluacion completa (10 configuraciones x 296 preguntas
# implica varios miles de llamadas al LLM juez).
TAMANO_MUESTRA = None

## 1. Cargar y normalizar las configuraciones a evaluar

Los dos archivos tienen formas distintas (uno agrupa 9 configuraciones bajo `"configuraciones"`, el otro es una sola configuracion "suelta"), asi que se homogenizan aqui en una sola lista.


In [41]:
import json
from pathlib import Path

# Todos los archivos de retrieval a incluir en la evaluacion.
# Para agregar un experimento futuro, solo añade su ruta a esta lista.
RUTAS_RESULTADOS = [
    "./validacion/resultados_retrieval.json",                     # 9 configuraciones dense-only
    "./validacion/resultados_retrieval_bge_m3_hibrido.json",      # BGE-m3 hibrido cs800, k=5
    "./validacion/resultados_retrieval_gemma_cs800_k3.json",      # experimento: Gemma cs800, k=3
    "./validacion/resultados_retrieval_bge_m3_hibrido_k10.json",  # experimento: BGE-m3 hibrido cs800, k=10
    "./validacion/resultados_retrieval_bge_m3_mejorado.json",      #experimento: BGE-m3 con re-ranker
    "./validacion/resultados_retrieval_openai_mejorado.json"       #experimento: openai con re-ranker
]


def _normalizar_configuracion(c: dict) -> dict:
    etiqueta = f"{c['modelo_embedding']}_cs{c['chunk_size']}"
    tipo_busqueda = c.get("tipo_busqueda", "densa")
    if tipo_busqueda != "densa":
        etiqueta += "_hibrido"
    if c.get("k") and c["k"] != 5:  # 5 es el k "base" de las configuraciones originales
        etiqueta += f"_k{c['k']}"

    return {
        "etiqueta": etiqueta,
        "modelo_embedding": c["modelo_embedding"],
        "chunk_size": c["chunk_size"],
        "tipo_busqueda": tipo_busqueda,
        "k": c.get("k"),
        "resultados": c["resultados"],
    }


def cargar_configuraciones(rutas=RUTAS_RESULTADOS):
    configuraciones = []
    for ruta in rutas:
        ruta = Path(ruta)
        if not ruta.exists():
            print(f"Aviso: no se encontro {ruta}")
            continue

        with open(ruta, "r", encoding="utf-8") as f:
            contenido = json.load(f)

        # Un archivo puede traer varias configuraciones (bajo "configuraciones")
        # o ser el mismo una sola configuracion suelta (tiene "resultados" arriba).
        crudas = contenido["configuraciones"] if "configuraciones" in contenido else [contenido]
        for c in crudas:
            configuraciones.append(_normalizar_configuracion(c))

    if TAMANO_MUESTRA:
        for c in configuraciones:
            c["resultados"] = c["resultados"][:TAMANO_MUESTRA]

    return configuraciones


configuraciones = cargar_configuraciones()
print(f"Configuraciones cargadas: {len(configuraciones)}")
for c in configuraciones:
    print(f"  {c['etiqueta']:50s} -> {len(c['resultados'])} preguntas (k={c['k']})")

Configuraciones cargadas: 14
  openai/text-embedding-3-large_cs300                -> 296 preguntas (k=5)
  openai/text-embedding-3-large_cs500                -> 296 preguntas (k=5)
  openai/text-embedding-3-large_cs800                -> 296 preguntas (k=5)
  BAAI/bge-m3_cs300                                  -> 296 preguntas (k=5)
  BAAI/bge-m3_cs500                                  -> 296 preguntas (k=5)
  BAAI/bge-m3_cs800                                  -> 296 preguntas (k=5)
  google/embeddinggemma-300m_cs300                   -> 296 preguntas (k=5)
  google/embeddinggemma-300m_cs500                   -> 296 preguntas (k=5)
  google/embeddinggemma-300m_cs800                   -> 296 preguntas (k=5)
  BAAI/bge-m3_con_reranking_cs800_hibrido            -> 296 preguntas (k=5)
  google/embeddinggemma-300m_cs800_k3                -> 296 preguntas (k=3)
  BAAI/bge-m3_cs800_hibrido_k10                      -> 296 preguntas (k=10)
  BAAI/bge-m3_cs800_hibrido                          -> 29

## 2. Metricas: precision y recall de contexto (con y sin LLM)

- **`LLMContextPrecisionWithReference`**: de los contextos recuperados, ¿cuales son relevantes para llegar a la respuesta esperada? (¿estamos trayendo ruido?). Usa el LLM juez.
- **`LLMContextRecall`**: de lo que hace falta para responder (la respuesta esperada, descompuesta en afirmaciones), ¿cuanto quedo cubierto por los contextos recuperados? (¿nos falta algo importante?). Usa el LLM juez.
- **`NonLLMContextPrecisionWithReference`** / **`NonLLMContextRecall`**: las mismas dos preguntas, pero sin LLM — comparan los contextos recuperados contra tu `evidencia_gold` con una medida de distancia de texto (Levenshtein via `rapidfuzz`). Gratis, deterministico, sin el sesgo de un juez LLM. Como ya tienes `evidencia_gold` en tu dataset, es casi gratis incluirlas como contraste/validacion de las metricas LLM.

No se incluyen `faithfulness` ni `answer_relevancy` porque requieren una respuesta generada, que todavia no existe en este flujo.


In [42]:
import sys
import types

# Parche: ragas intenta importar ChatVertexAI de langchain_community aunque
# no lo usemos aqui. En versiones recientes de langchain_community ese modulo
# ya no existe, lo cual rompe el import de ragas por completo. Como no
# necesitamos VertexAI para nada, le damos un modulo de relleno para que
# ragas pueda terminar de importarse.
if "langchain_community.chat_models.vertexai" not in sys.modules:
    try:
        import langchain_community.chat_models.vertexai  # noqa: F401
    except ModuleNotFoundError:
        modulo_falso = types.ModuleType("langchain_community.chat_models.vertexai")

        class ChatVertexAI:  # stub, nunca se instancia de verdad
            pass

        modulo_falso.ChatVertexAI = ChatVertexAI
        sys.modules["langchain_community.chat_models.vertexai"] = modulo_falso
        print("Aviso: se aplico un parche al import roto de langchain_community.chat_models.vertexai (no afecta esta evaluacion, no usamos VertexAI).")

import os
from langchain_openai import ChatOpenAI
from ragas.llms import LangchainLLMWrapper
from ragas.run_config import RunConfig

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    raise RuntimeError("Define la variable de entorno OPENROUTER_API_KEY")

MODELO_JUEZ = "openai/gpt-4.1-mini"
MODELOS_JUEZ_RESPALDO = ["google/gemini-2.5-flash", "deepseek/deepseek-chat"]

llm_juez_base = ChatOpenAI(
    model=MODELO_JUEZ,
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    temperature=0,
    extra_body={"models": MODELOS_JUEZ_RESPALDO},
)
evaluator_llm = LangchainLLMWrapper(llm_juez_base)

run_config = RunConfig(timeout=120, max_retries=8, max_wait=90, max_workers=4)

C:\Users\gomex\AppData\Local\Temp\ipykernel_37080\3355197474.py:41: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(llm_juez_base)


In [43]:
from ragas.metrics import (
    LLMContextPrecisionWithReference,
    LLMContextRecall,
    NonLLMContextPrecisionWithReference,
    NonLLMContextRecall,
)

METRICAS = [
    LLMContextPrecisionWithReference(llm=evaluator_llm),
    LLMContextRecall(llm=evaluator_llm),
    NonLLMContextPrecisionWithReference(),
    NonLLMContextRecall(),
]

print("Metricas a calcular:", [m.name for m in METRICAS])

Metricas a calcular: ['llm_context_precision_with_reference', 'context_recall', 'non_llm_context_precision_with_reference', 'non_llm_context_recall']


C:\Users\gomex\AppData\Local\Temp\ipykernel_37080\3493589588.py:1: DeprecationWarning: Importing LLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithReference
  from ragas.metrics import (
C:\Users\gomex\AppData\Local\Temp\ipykernel_37080\3493589588.py:1: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import (
C:\Users\gomex\AppData\Local\Temp\ipykernel_37080\3493589588.py:1: DeprecationWarning: Importing NonLLMContextPrecisionWithReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import NonLLMContextPrecisionWithReference


## 3. Evaluar cada configuracion (con checkpoint y reintentos)

Si una configuracion ya fue evaluada antes (existe su checkpoint en `RUTA_CHECKPOINTS`), se reutiliza en vez de volver a gastar llamadas al LLM. Si `evaluate()` falla por completo (por ejemplo, una caida prolongada que agota los reintentos internos de RAGAS), se reintenta la configuracion completa unas cuantas veces mas antes de rendirse y seguir con la siguiente (respaldo, capa 3).


In [44]:
from ragas import evaluate, EvaluationDataset


def construir_dataset(resultados):
    filas = [
        {
            "user_input": r["pregunta"],
            "reference": r.get("respuesta_esperada") or "",
            "retrieved_contexts": r.get("contextos_recuperados") or [],
            "reference_contexts": r.get("evidencia_gold") or [],
        }
        for r in resultados
    ]
    return EvaluationDataset.from_list(filas)

In [45]:
import time
import pandas as pd


def evaluar_configuracion(config, intentos_externos=3, espera_entre_intentos=60):
    ruta_checkpoint = RUTA_CHECKPOINTS / f"{config['etiqueta']}.csv"

    if ruta_checkpoint.exists():
        print(f"  [{config['etiqueta']}] ya evaluado antes, se reutiliza el checkpoint.")
        return pd.read_csv(ruta_checkpoint)

    dataset = construir_dataset(config["resultados"])

    for intento in range(1, intentos_externos + 1):
        try:
            resultado = evaluate(
                dataset=dataset,
                metrics=METRICAS,
                llm=evaluator_llm,
                run_config=run_config,
                raise_exceptions=False,  # una fila que falle da NaN, no tumba todo el batch
                show_progress=True,
            )
            df = resultado.to_pandas()
            df.to_csv(ruta_checkpoint, index=False, encoding="utf-8")
            print(f"  [{config['etiqueta']}] listo, checkpoint guardado en {ruta_checkpoint}")
            return df
        except Exception as e:
            print(f"  [{config['etiqueta']}] intento {intento}/{intentos_externos} fallo: {e}")
            if intento < intentos_externos:
                print(f"  Esperando {espera_entre_intentos}s antes de reintentar la configuracion completa...")
                time.sleep(espera_entre_intentos)
            else:
                print(f"  [{config['etiqueta']}] se agotaron los reintentos, se omite esta configuracion.")
                return None

In [46]:
resultados_por_configuracion = {}

for config in configuraciones:
    print(f"\n=== Evaluando: {config['etiqueta']} ({len(config['resultados'])} preguntas) ===")
    df = evaluar_configuracion(config)
    if df is not None:
        df["configuracion"] = config["etiqueta"]
        df["modelo_embedding"] = config["modelo_embedding"]
        df["chunk_size"] = config["chunk_size"]
        df["tipo_busqueda"] = config["tipo_busqueda"]
        resultados_por_configuracion[config["etiqueta"]] = df

print(f"\nConfiguraciones evaluadas con exito: {len(resultados_por_configuracion)} / {len(configuraciones)}")


=== Evaluando: openai/text-embedding-3-large_cs300 (296 preguntas) ===
  [openai/text-embedding-3-large_cs300] ya evaluado antes, se reutiliza el checkpoint.

=== Evaluando: openai/text-embedding-3-large_cs500 (296 preguntas) ===
  [openai/text-embedding-3-large_cs500] ya evaluado antes, se reutiliza el checkpoint.

=== Evaluando: openai/text-embedding-3-large_cs800 (296 preguntas) ===
  [openai/text-embedding-3-large_cs800] ya evaluado antes, se reutiliza el checkpoint.

=== Evaluando: BAAI/bge-m3_cs300 (296 preguntas) ===
  [BAAI/bge-m3_cs300] ya evaluado antes, se reutiliza el checkpoint.

=== Evaluando: BAAI/bge-m3_cs500 (296 preguntas) ===
  [BAAI/bge-m3_cs500] ya evaluado antes, se reutiliza el checkpoint.

=== Evaluando: BAAI/bge-m3_cs800 (296 preguntas) ===
  [BAAI/bge-m3_cs800] ya evaluado antes, se reutiliza el checkpoint.

=== Evaluando: google/embeddinggemma-300m_cs300 (296 preguntas) ===
  [google/embeddinggemma-300m_cs300] ya evaluado antes, se reutiliza el checkpoint.



Evaluating: 100%|██████████| 1184/1184 [25:26<00:00,  1.29s/it]


  [BAAI/bge-m3_con_reranking_cs800_hibrido] listo, checkpoint guardado en validacion\checkpoints_evaluacion\BAAI\bge-m3_con_reranking_cs800_hibrido.csv

=== Evaluando: google/embeddinggemma-300m_cs800_k3 (296 preguntas) ===
  [google/embeddinggemma-300m_cs800_k3] ya evaluado antes, se reutiliza el checkpoint.

=== Evaluando: BAAI/bge-m3_cs800_hibrido_k10 (296 preguntas) ===
  [BAAI/bge-m3_cs800_hibrido_k10] ya evaluado antes, se reutiliza el checkpoint.

=== Evaluando: BAAI/bge-m3_cs800_hibrido (296 preguntas) ===
  [BAAI/bge-m3_cs800_hibrido] ya evaluado antes, se reutiliza el checkpoint.

=== Evaluando: openai/text-embedding-3-large_cs800_hibrido (296 preguntas) ===
  [openai/text-embedding-3-large_cs800_hibrido] ya evaluado antes, se reutiliza el checkpoint.

Configuraciones evaluadas con exito: 14 / 14


## 4. Consolidar y guardar

Un CSV detallado (una fila por pregunta x configuracion, para poder inspeccionar casos puntuales) y un CSV resumen (promedio de cada metrica por configuracion, para comparar de un vistazo).


In [47]:
if resultados_por_configuracion:
    df_detallado = pd.concat(resultados_por_configuracion.values(), ignore_index=True)
else:
    df_detallado = pd.DataFrame()

df_detallado.to_csv(RUTA_SALIDA_DETALLE, index=False, encoding="utf-8")
print(f"Detalle por pregunta guardado en {RUTA_SALIDA_DETALLE} ({len(df_detallado)} filas)")

# Columnas de metricas: todo lo que no sea el texto de entrada/salida ni las
# columnas de identificacion de la configuracion que agregamos nosotros.
COLUMNAS_NO_METRICA = {
    "user_input", "reference", "retrieved_contexts", "reference_contexts",
    "configuracion", "modelo_embedding", "chunk_size", "tipo_busqueda",
}
columnas_metricas = [c for c in df_detallado.columns if c not in COLUMNAS_NO_METRICA]

if not df_detallado.empty:
    resumen = (
        df_detallado.groupby(["configuracion", "modelo_embedding", "chunk_size", "tipo_busqueda"])[columnas_metricas]
        .mean()
        .reset_index()
        .sort_values("configuracion")
    )
    resumen.to_csv(RUTA_SALIDA_RESUMEN, index=False, encoding="utf-8")
    print(f"Resumen por configuracion guardado en {RUTA_SALIDA_RESUMEN}")
else:
    resumen = pd.DataFrame()
    print("No hay resultados para resumir.")

resumen

Detalle por pregunta guardado en ./validacion/evaluacion_ragas_detalle.csv (4144 filas)
Resumen por configuracion guardado en ./validacion/evaluacion_ragas_resumen.csv


,configuracion,modelo_embedding,chunk_size,tipo_busqueda,llm_context_precision_with_reference,context_recall,non_llm_context_precision_with_reference,non_llm_context_recall
0,BAAI/bge-m3_con_reranking_cs800_hibrido,BAAI/bge-m3_con_reranking,800,hibrida_dense_sparse_rrf,0.688298,0.736036,0.015878,0.020270
1,BAAI/bge-m3_cs300,BAAI/bge-m3,300,densa,0.614241,0.636196,0.059741,0.076689
2,BAAI/bge-m3_cs500,BAAI/bge-m3,500,densa,0.603674,0.699043,0.024212,0.033784
3,BAAI/bge-m3_cs800,BAAI/bge-m3,800,densa,0.632639,0.700760,0.016610,0.020270
4,BAAI/bge-m3_cs800_hibrido,BAAI/bge-m3,800,hibrida_dense_sparse_rrf_rerank,0.701276,0.735980,0.015878,0.020270
5,BAAI/bge-m3_cs800_hibrido_k10,BAAI/bge-m3,800,hibrida_dense_sparse_rrf,0.634940,0.809009,0.015878,0.020270
6,google/embeddinggemma-300m_cs300,google/embeddinggemma-300m,300,densa,0.692835,0.714855,0.074718,0.090203
7,google/embeddinggemma-300m_cs500,google/embeddinggemma-300m,500,densa,0.697438,0.746284,0.034037,0.047297
8,google/embeddinggemma-300m_cs800,google/embeddinggemma-300m,800,densa,0.665419,0.746903,0.014330,0.023649
9,google/embeddinggemma-300m_cs800_k3,google/embeddinggemma-300m,800,densa,0.652027,0.674268,0.015766,0.023649


## Notas finales

- **Resiliencia en 3 capas**: fallback de modelos de OpenRouter (nivel de peticion) -> reintentos con backoff de RAGAS (`RunConfig`, nivel de llamada individual) -> reintentos + checkpoint por configuracion (nivel de notebook, este archivo). Si el kernel se cae o OpenRouter tiene una caida larga, puedes volver a correr este notebook y las configuraciones ya evaluadas se recuperan del checkpoint en `RUTA_CHECKPOINTS` sin gastar llamadas de nuevo.
- **Como leer los resultados**: si `llm_context_precision_with_reference` y `non_llm_context_precision_with_reference` (o los dos de recall) difieren mucho para una misma configuracion, vale la pena mirar de cerca esa configuracion — puede ser una señal de que el juez LLM esta siendo demasiado estricto/laxo, o de que `evidencia_gold` no calza literalmente con el texto de tus chunks (por ejemplo, si el chunking corto una oracion de la evidencia a la mitad).
- Este notebook asume que ya corriste los dos notebooks de retrieval y tienes `resultados_retrieval.json` y `resultados_retrieval_bge_m3_hibrido.json` en `./validacion/`. Si solo tienes uno de los dos, el notebook sigue funcionando (evalua lo que encuentre).
- Antes de lanzar la evaluacion completa (10 configuraciones x 296 preguntas), corre una vez con `TAMANO_MUESTRA = 15` o `20` para confirmar que todo funciona y hacerte una idea del costo/tiempo real, y luego pon `TAMANO_MUESTRA = None` para la corrida completa.
- Siguiente paso natural: generar respuestas con un LLM sobre `contextos_recuperados` y añadir `faithfulness` / `answer_relevancy` para evaluar tambien la generacion, no solo el retrieval.
